# Detection Analysis

This notebook demonstrates how to analyze efficiency, purity, and stamps for the transients and variables in the subtraction pipeline.

Assumes OpenUniverse2024 for truth data to calculate efficiency and purity.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import matplotlib.patches as mpatches

from astropy.io import fits
from astropy.table import Table
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.visualization import ZScaleInterval
from astropy.wcs.utils import skycoord_to_pixel, pixel_to_skycoord
interval = ZScaleInterval()

In [ ]:
# Change this to a CSV file with a list of the {science,template}_{pointing,band,sca}
data_record_path = "../tests/test_one_data_record.csv"
# data_record_path = "../tests/test_ten_data_records.csv"

In [ ]:
def get_difference_id(science_id, template_id):
    _prefixed_science = {f"science_{k}": v for k, v in science_id.items()}
    _prefixed_template = {f"template_{k}": v for k, v in template_id.items()}
    difference_id = {**_prefixed_science, **_prefixed_template}
    return difference_id

def load_fits(image_path, return_hdr=True, return_data=True, return_wcs=True, hdu_index=0):
    with fits.open(image_path) as hdul:
        hdr = hdul[hdu_index].header if return_hdr or return_wcs else None
        wcs = WCS(hdr) if return_wcs else None
        data = hdul[hdu_index].data if return_data else None
    return hdr, wcs, data

def load_table(table_path):
    table = Table.read(table_path, format='ascii').to_pandas()
    return table

def xy_in_image(x, y, width, height, offset=0):
    return (0 + offset <= x) & (x < width - offset) & (0 + offset <= y) & (y < height - offset)

def project_source_xy_to_radec(source, wcs, frame="fk5", origin=1):
    """

    Parameters
    ----------
    frame: The OpenUniverse sims are actually in FK5/J2000 rather than ICRS
    origin: The center of the first pixel is 1, 1 for both source extractor and the OpenUniverse truth catalog.
    """
    if "x_peak" in source.keys():
        coord = pixel_to_skycoord(source["x_peak"], source["y_peak"], wcs, origin=origin)
        coord = coord.transform_to(frame)
        source["ra"] = coord.ra.to(u.deg).value
        source["dec"] = coord.dec.to(u.deg).value
    elif "X_IMAGE" in source.keys():
        coord = pixel_to_skycoord(source["X_IMAGE"], source["Y_IMAGE"], wcs, origin=origin)
        coord = coord.transform_to(frame)
        source["ra"] = coord.ra.to(u.deg).value
        source["dec"] = coord.dec.to(u.deg).value

    return source

def project_source_radec_to_xy(source, science_wcs, template_wcs, score_wcs, difference_wcs, image_width, image_height, origin=1,
                               frame="fk5", ra_col="ra", dec_col="dec"):
    """

    Parameters
    ----------
    frame: The OpenUniverse sims are actually in FK5/J2000 rather than ICRS
    origin: The center of the first pixel is 1, 1 for both source extractor and the OpenUniverse truth catalog.
    """
    source_copy = source.copy()

    radec = SkyCoord(source_copy[ra_col], source_copy[dec_col], frame=frame, unit='deg')
    
    x_in_science, y_in_science = skycoord_to_pixel(radec, science_wcs, origin=origin)
    source_copy['x_in_science'] = x_in_science
    source_copy['y_in_science'] = y_in_science

    x_in_template, y_in_template = skycoord_to_pixel(radec, template_wcs, origin=origin)
    source_copy['x_in_template'] = x_in_template
    source_copy['y_in_template'] = y_in_template

    x_in_score, y_in_score = skycoord_to_pixel(radec, score_wcs, origin=origin)
    source_copy['x_in_score'] = x_in_score
    source_copy['y_in_score'] = y_in_score

    x_in_difference, y_in_difference = skycoord_to_pixel(radec, difference_wcs, origin=origin)
    source_copy['x_in_difference'] = x_in_difference
    source_copy['y_in_difference'] = y_in_difference
    
    return source_copy

def show_image(ax, image, title=None, xlabel=None, ylabel=None, axis_off=False, zscale=True):
    if zscale:
        image = interval(image)
    ax.imshow(image, origin='lower', cmap='gray')
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if axis_off:
        ax.set_axis_off()

def add_circles(ax, x, y, radius=0.5, color='red', alpha=1):
    for xp, yp in zip(x, y):
        circle = Circle((xp, yp), radius=radius, color=color, alpha=alpha, fill=False)
        ax.add_patch(circle)

def crop_image(image, cr, cc, half_r=50, half_c=50, fill_edge=True, fill_value=np.nan):
    cr, cc = np.floor(cr).astype(int), np.floor(cc).astype(int)
    if  cr <= 0 or image.shape[0] - 1 <= cr or cc <= 0 or image.shape[1] - 1 <= cc:
        raise ValueError(f"Image center at (cr={cr}, cc={cc}) is out of bounds. "
                             f"Valid cr range: [{0}, {image.shape[0]}], y range: [{0}, {image.shape[1]}].")

    half_r, half_c = np.floor(half_r).astype(int), np.floor(half_c).astype(int)
    r_left = min(half_r, cr)
    r_right = min(half_r, image.shape[0] - 1 - cr)
    c_left = min(half_c, cc)
    c_right = min(half_c, image.shape[1] - 1 - cc)

    image_slice = image[cr - r_left: cr + r_right + 1, cc - c_left: cc + c_right + 1].copy()

    if fill_edge:
        cutout = np.full( (2 * half_r + 1, 2 * half_c + 1), fill_value)
        cutout[half_r - r_left: half_r + r_right + 1, half_c - c_left: half_c + c_right + 1] = image_slice
    else:
        cutout = image_slice
    
    return cutout

In [ ]:
def analyze_detection(science_band, science_pointing, science_sca,
                      template_band, template_pointing, template_sca,
                      output_dir,
                      verbose=True):
    INPUT_IMAGE_PATTERN = ("/global/cfs/cdirs/lsst/shared/external/roman-desc-sims/Roman_data"
                                    "/RomanTDS/images/simple_model/{band}/{pointing}/Roman_TDS_simple_model_{band}_{pointing}_{sca}.fits.gz")
    INPUT_TRUTH_PATTERN = ("/global/cfs/cdirs/lsst/shared/external/roman-desc-sims/Roman_data"
                                 "/RomanTDS/truth/{band}/{pointing}/Roman_TDS_index_{band}_{pointing}_{sca}.txt")
    
    DIFF_IMAGE_PREFIX = 'decorr_diff_'
    DIFF_SCORE_PREFIX = "score_"
    DIFF_DETECTION_PREFIX = 'detection_'
    DIFF_TRUTH_PREFIX = 'truth_'
    DIFF_DETECTION_PREFIX = "detection_"
    SCORE_DETECTION_PREFIX = "score_detection_"
    TRANSIENTS_TO_DETECTION_PREFIX = 'transients_to_detection_'
    TRANSIENTS_TO_SCORE_DETECTION_PREFIX = 'transients_to_score_detection_'
    DETECTION_TO_TRANSIENTS_PREFIX = 'detection_to_transients_'
    SCORE_DETECTION_TO_TRANSIENTS_PREFIX = "score_detection_to_transients_"
    TRANSIENTS_TO_CLEANED_DETECTION_PREFIX = 'transients_to_cleaned_detection_'
    TRANSIENTS_TO_CLEANED_SCORE_DETECTION_PREFIX = 'transients_to_cleaned_score_detection_'
    CLEANED_DETECTION_TO_TRANSIENTS_PREFIX = 'cleaned_detection_to_transients_'
    CLEANED_SCORE_DETECTION_TO_TRANSIENTS_PREFIX = "cleaned_score_detection_to_transients_"
    
    DIFF_PATTERN = '{science_band}_{science_pointing}_{science_sca}_-_{template_band}_{template_pointing}_{template_sca}'
        
    IMAGE_WIDTH = 4088
    IMAGE_HEIGHT = 4088

    science_id = {'band': row['science_band'], 'pointing': row['science_pointing'], 'sca': row['science_sca']}
    template_id = {'band': row['template_band'], 'pointing': row['template_pointing'], 'sca': row['template_sca']}
    difference_id = get_difference_id(science_id, template_id)
    diff_pattern = DIFF_PATTERN.format(**difference_id)
    
    science_image_path = INPUT_IMAGE_PATTERN.format(**science_id)
    science_truth_path = INPUT_TRUTH_PATTERN.format(**science_id)
    
    template_image_path = INPUT_IMAGE_PATTERN.format(**template_id)
    template_truth_path = INPUT_TRUTH_PATTERN.format(**template_id)

    score_image_path = os.path.join(output_dir, diff_pattern, DIFF_SCORE_PREFIX + diff_pattern + '.fits')
    difference_image_path = os.path.join(output_dir, diff_pattern, DIFF_IMAGE_PREFIX + diff_pattern + '.fits')
    transients_to_detection_path = os.path.join(output_dir, diff_pattern, TRANSIENTS_TO_DETECTION_PREFIX + diff_pattern + '.ecsv')
    detection_to_transients_path = os.path.join(output_dir, diff_pattern, DETECTION_TO_TRANSIENTS_PREFIX + diff_pattern + '.ecsv')
    transients_to_score_detection_path = os.path.join(output_dir, diff_pattern, TRANSIENTS_TO_SCORE_DETECTION_PREFIX + diff_pattern + '.ecsv')
    score_detection_to_transients_path = os.path.join(output_dir, diff_pattern, SCORE_DETECTION_TO_TRANSIENTS_PREFIX + diff_pattern + '.ecsv')
    transients_to_cleaned_detection_path = os.path.join(output_dir, diff_pattern, TRANSIENTS_TO_CLEANED_DETECTION_PREFIX + diff_pattern + '.ecsv')
    cleaned_detection_to_transients_path = os.path.join(output_dir, diff_pattern, CLEANED_DETECTION_TO_TRANSIENTS_PREFIX + diff_pattern + '.ecsv')
    transients_to_cleaned_score_detection_path = os.path.join(output_dir, diff_pattern, TRANSIENTS_TO_CLEANED_SCORE_DETECTION_PREFIX + diff_pattern + '.ecsv')
    cleaned_score_detection_to_transients_path = os.path.join(output_dir, diff_pattern, CLEANED_SCORE_DETECTION_TO_TRANSIENTS_PREFIX + diff_pattern + '.ecsv')
   
    _, science_wcs, science_image = load_fits(science_image_path, hdu_index=1)
    _, template_wcs, template_image = load_fits(template_image_path, hdu_index=1)
    _, score_wcs, score_image = load_fits(score_image_path, hdu_index=0)
    _, difference_wcs, difference_image = load_fits(difference_image_path, hdu_index=0)
    
    transients_to_detection = Table.read(transients_to_detection_path, format="ascii.ecsv")
    transients_to_detection = project_source_radec_to_xy(transients_to_detection, science_wcs, template_wcs, score_wcs, difference_wcs, IMAGE_WIDTH, IMAGE_HEIGHT, ra_col="object_ra", dec_col="object_dec")
    transients_to_score_detection = Table.read(transients_to_score_detection_path, format="ascii.ecsv")
    transients_to_score_detection = project_source_radec_to_xy(transients_to_score_detection, science_wcs, template_wcs, score_wcs, difference_wcs, IMAGE_WIDTH, IMAGE_HEIGHT, ra_col="object_ra", dec_col="object_dec")
    detection_to_transients = Table.read(detection_to_transients_path, format="ascii.ecsv")
    detection_to_transients = project_source_xy_to_radec(detection_to_transients, difference_wcs)
    detection_to_transients = project_source_radec_to_xy(detection_to_transients, science_wcs, template_wcs, score_wcs, difference_wcs, IMAGE_WIDTH, IMAGE_HEIGHT)
    score_detection_to_transients = Table.read(score_detection_to_transients_path, format="ascii.ecsv")
    score_detection_to_transients = project_source_xy_to_radec(score_detection_to_transients, score_wcs)
    score_detection_to_transients = project_source_radec_to_xy(score_detection_to_transients, science_wcs, template_wcs, score_wcs, difference_wcs, IMAGE_WIDTH, IMAGE_HEIGHT)

    transients_to_cleaned_detection = Table.read(transients_to_cleaned_detection_path, format="ascii.ecsv")
    transients_to_cleaned_detection = project_source_radec_to_xy(transients_to_cleaned_detection, science_wcs, template_wcs, score_wcs, difference_wcs, IMAGE_WIDTH, IMAGE_HEIGHT, ra_col="object_ra", dec_col="object_dec")
    transients_to_cleaned_score_detection = Table.read(transients_to_cleaned_score_detection_path, format="ascii.ecsv")
    transients_to_cleaned_score_detection = project_source_radec_to_xy(transients_to_cleaned_score_detection, science_wcs, template_wcs, score_wcs, difference_wcs, IMAGE_WIDTH, IMAGE_HEIGHT, ra_col="object_ra", dec_col="object_dec")
    cleaned_detection_to_transients = Table.read(cleaned_detection_to_transients_path, format="ascii.ecsv")
    cleaned_detection_to_transients = project_source_xy_to_radec(cleaned_detection_to_transients, difference_wcs)
    cleaned_detection_to_transients = project_source_radec_to_xy(cleaned_detection_to_transients, science_wcs, template_wcs, score_wcs, difference_wcs, IMAGE_WIDTH, IMAGE_HEIGHT)
    cleaned_score_detection_to_transients = Table.read(cleaned_score_detection_to_transients_path)
    cleaned_score_detection_to_transients = project_source_xy_to_radec(cleaned_score_detection_to_transients, score_wcs)
    cleaned_score_detection_to_transients = project_source_radec_to_xy(cleaned_score_detection_to_transients, science_wcs, template_wcs, score_wcs, difference_wcs, IMAGE_WIDTH, IMAGE_HEIGHT)

    data_products = {}
    data_products['science_image'] = science_image
    data_products['template_image'] = template_image
    data_products['score_image'] = score_image
    data_products['difference_image'] = difference_image

    data_products['transients_to_detection'] = transients_to_detection
    data_products['detection_to_transients'] = detection_to_transients
    data_products['transients_to_score_detection'] = transients_to_score_detection
    data_products['score_detection_to_transients'] = score_detection_to_transients
    data_products['transients_to_cleaned_detection'] = transients_to_cleaned_detection
    data_products['cleaned_detection_to_transients'] = cleaned_detection_to_transients
    data_products['transients_to_cleaned_score_detection'] = transients_to_cleaned_score_detection
    data_products['cleaned_score_detection_to_transients'] = cleaned_score_detection_to_transients
    
    return data_products

In [ ]:
def visualize_sources_on_image(dp, detection_type="transients_to_cleaned_score_detection", should_detect=True, did_detect=False):
    matched_transients = dp['transients_to_score_detection'][dp['transients_to_score_detection']["matched_status"]].copy()
    
    fig, ax = plt.subplots(1, 4, figsize=(15, 10))

    cat = dp[detection_type]
    if should_detect:
        cat = cat[cat["should_detect"]]
    if did_detect:
        cat = cat[cat["did_detect"]]
    
    show_image(ax[0], dp['template_image'], title='Template', axis_off=True)
    show_image(ax[1], dp['science_image'], title='Science', axis_off=True)
    show_image(ax[2], dp['score_image'], title='Score')
    show_image(ax[3], dp['difference_image'], title='Difference')

    for ta in ax:
        add_circles(ta, cat["x_in_difference"] - 1, cat["y_in_difference"] - 1, radius=15, color="red")
        add_circles(ta, matched_transients["x_in_difference"] - 1, matched_transients["y_in_difference"] - 1, radius=25, color="blue")
    
    red_patch = mpatches.Patch(color='red', label='Science Truth')
    blue_patch = mpatches.Patch(color='blue', label='Detected Truth')
    fig.legend(handles=[red_patch, blue_patch], loc='lower center', ncol=2, fontsize=12, bbox_to_anchor=(0.5, 0.2))
    
    fig.subplots_adjust(bottom=0)

In [ ]:
def visualize_stamps(dp, figsize=None, detection_type="transients_to_score_detection", stamp_radius=20, should_detect=False, did_detect=False, only_detected=False, debug=False):

    rows = dp[detection_type]
    if should_detect:
        rows = rows[rows["should_detect"]]
    if did_detect:
        rows = rows[rows["did_detect"]]
        
    num_rows = len(rows)

    crop_kwargs = {"half_r": stamp_radius, "half_c": stamp_radius, "fill_edge": True, "fill_value": np.nan}

    image_names = ["template", "science", "score", "difference"]
    num_image_names = len(image_names)
    
    if figsize is None:
        figsize = (2.5 * num_image_names, 3 * num_rows)
    fig, axes = plt.subplots(num_rows, num_image_names, figsize=figsize)
    
    for ax, row in zip(axes, rows):
        if only_detected and row["matched_status"] is False:
            continue

        titles = [
            f"ID: {row["object_id"]}",
            f"Realized Flux: {row['realized_flux']:.0f}",
            f"Peak Value: {row['peak_value']:0.2f}",
            f"Matched: {row['matched_status']}",
        ]

        for ta, imn, title in zip(ax, image_names, titles):
            cr = row[f"y_in_{imn}"] - 1
            cc = row[f"x_in_{imn}"] - 1

            if cr < 0 or cc < 0:
                print(f"stamp includes pixels outside of image ({cr}, {cc}), skipping")
                continue

            cutout = crop_image(dp[f"{imn}_image"], cr=cr, cc=cc, **crop_kwargs)
            ta.imshow(interval(cutout), cmap='gray', origin='lower')
            ta.set_title(title)

            # Coord system of science, score, and difference should be the same
            # but in principle, we could have warped science to template or something like that
            # so we might as well keep track of these separately.
            # We cropped by integer pixel, so let's put the fraction part back in to get the circle position exactly right
            r_center = row[f"y_in_{imn}"] % 1 + crop_kwargs["half_r"]
            c_center = row[f"x_in_{imn}"] % 1 + crop_kwargs["half_c"]
            if debug:
                print("object x, y: ", r_center, c_center)
            add_circles(ta, [r_center], [c_center], radius=5, color='red')
            ta.set_axis_off()

In [ ]:
data_records = pd.read_csv(data_record_path)

In [ ]:
row = data_records.iloc[0]
output_dir = "/global/homes/w/wmwv/Roman/pipeline/dia_out_dir"

In [ ]:
dp = analyze_detection(row["science_band"], row["science_pointing"], row["science_sca"],
                       row["template_band"], row["template_pointing"], row["template_sca"],
                       output_dir)

# Need to actually calculate an error at some point, but let's start with this simple threshold
# Assume no sky noise, all shot noise, 
snr_threshold = 10
realized_flux_threshold = snr_threshold ** 2
peak_value_threshold = 25  # I don't know what this should be

detection_type = "transients_to_cleaned_score_detection"

should_detect = np.abs(dp[detection_type]["realized_flux"]) > realized_flux_threshold
score_should_detect = np.abs(dp["cleaned_score_detection_to_transients"]["realized_flux"]) > realized_flux_threshold
score_did_detect = np.abs(dp["cleaned_score_detection_to_transients"]["peak_value"]) > peak_value_threshold

dp[detection_type]["should_detect"] = should_detect
dp["cleaned_score_detection_to_transients"]["should_detect"] = score_should_detect
dp["cleaned_score_detection_to_transients"]["did_detect"] = score_did_detect

In [ ]:
efficiency = dp[detection_type]["matched_status"][should_detect].sum() / len(dp[detection_type][should_detect])
# We get credit for detecting even if it doesn't pass the SNR threshold for the injected realized_flux.
purity = dp['cleaned_score_detection_to_transients'][score_did_detect]["matched_status"].sum() / len(dp["cleaned_score_detection_to_transients"][score_did_detect])

print(f"Efficiency: {efficiency:0.3f},  Purity: {purity:0.3f}")

In [ ]:
dp["cleaned_score_detection_to_transients"][score_did_detect]

In [ ]:
visualize_sources_on_image(dp)

In [ ]:
visualize_stamps(dp, detection_type="transients_to_cleaned_score_detection", should_detect=True)

In [ ]:
visualize_stamps(dp, detection_type="cleaned_score_detection_to_transients", did_detect=False)

In [ ]:
dp["cleaned_score_detection_to_transients"]["peak_value"]